# Episode 31: Reasoning

An agent that returns *only* an answer gives you nothing to check. Was the answer reasoned, or guessed? You can't tell.

**In this episode you'll build:** an agent that shows its work — chain-of-thought captured as **structured, inspectable output**.

## Why structured reasoning

Asking a model to "think step by step" is a well-known accuracy trick. But if those steps come back as free-form prose, your code can't *use* them.

Combine chain-of-thought with **structured output** (Episode 9): a `response_schema` with an explicit `steps` field. Now the reasoning is:

- **inspectable** — your code can read each step
- **auditable** — log the steps alongside the answer
- **routable** — a `confidence` field lets you escalate uncertain answers to a human

You get the accuracy of CoT *and* a machine-readable trace.

## A reasoning schema

The schema makes the contract explicit: an ordered list of `steps`, the final `answer`, and a `confidence` score. Passing it as `response_schema` forces the model to fill every field.

In [1]:
from dotenv import load_dotenv

load_dotenv()

from typing import Annotated

from pydantic import BaseModel, Field

from autogen.beta import Agent
from autogen.beta.config import OpenAIConfig

config = OpenAIConfig(model="gpt-4.1-mini", temperature=0)


class Reasoning(BaseModel):
    steps: Annotated[list[str], Field(description="Each intermediate reasoning step, in order")]
    answer: Annotated[str, Field(description="The final answer")]
    confidence: Annotated[float, Field(description="Confidence from 0.0 to 1.0", ge=0.0, le=1.0)]


reasoner = Agent(
    "reasoner",
    prompt="Think step by step. Show every intermediate step before giving the answer.",
    config=config,
    response_schema=Reasoning,
)

reply = await reasoner.ask(
    "A train leaves at 2:15 PM and arrives at 4:45 PM. The return trip is 30 minutes "
    "longer. How long is the return trip, in minutes?"
)
result = await reply.content()

print("steps:")
for i, step in enumerate(result.steps, 1):
    print(f"  {i}. {step}")
print("answer:    ", result.answer)
print("confidence:", result.confidence)


steps:
  1. Calculate the duration of the initial trip by finding the difference between 4:45 PM and 2:15 PM.
  2. From 2:15 PM to 4:15 PM is 2 hours, which is 120 minutes.
  3. From 4:15 PM to 4:45 PM is 30 minutes.
  4. Total duration of the initial trip is 120 + 30 = 150 minutes.
  5. The return trip is 30 minutes longer than the initial trip.
  6. Add 30 minutes to 150 minutes to find the return trip duration.
  7. 150 + 30 = 180 minutes.
answer:     180 minutes
confidence: 1.0


## What this buys you

`result` is a typed `Reasoning` object — your code can act on every field:

- **Verify the steps.** A right answer reached by wrong reasoning is a bug waiting to happen. With the steps in hand, you (or another agent) can check the *path*, not just the destination.
- **Route on confidence.** `if result.confidence < 0.7: escalate_to_human(result)` — low-confidence answers get a second look instead of shipping silently.
- **Audit.** Log `result.steps` next to the answer. When something goes wrong in production, you have the full reasoning trace, not just the verdict.

## Additional content

**Reasoning vs. reasoning models.** This pattern works with *any* model — it prompts ordinary CoT and captures it. It's complementary to provider "reasoning" models: those think internally; this makes the thinking an explicit, typed part of the response you control.

**Keep the schema honest.** If you add a `confidence` field but never act on it, it's noise. Only ask for structure you'll actually use — every field is something the model must spend tokens filling.

**Pair it with redundancy.** Episode 30's panel plus this episode's schema: run several reasoners, compare not just their answers but their *steps* — agreement on the path is a stronger signal than agreement on the answer alone.

## What's next

You've seen events surface throughout Group 6 — telemetry, observers, audit logs. Episode 32 goes to the source: the **event stream** itself, and how to program against it directly.